In [ ]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from datetime import datetime
from pathlib import Path
import json
import time
from ultralytics import YOLO
from examwatch.monitoring_logic import (
    EventTracker, face_measurements, calibration_baseline, classify_measurements,
)

# Thresholds are starting points to validate using your own labeled clips.
CAMERA_INDEX = 0
MAX_FACES = 5
CALIBRATION_FRAMES = 150  # Valid single-face samples, not total camera frames.
MIN_EVENT_DURATION = 1.0
RECOVERY_PERIOD = 0.5
FACE_ABSENCE_SECONDS = 3.0
HIGH_ALERT_SECONDS = 3.0
HEAD_THRESHOLD = 0.15
PITCH_THRESHOLD = 0.15
GAZE_THRESHOLD = 0.10
OBJECT_CONFIDENCE = 0.40
OBJECT_CLASSES = {67: "CELL PHONE", 73: "BOOK", 74: "CLOCK"}
ALLOWED_OBJECTS = {"CLOCK"}
MODEL_DIR = Path("models")
SCREENSHOT_DIR = Path("outputs/screenshots")
REPORT_DIR = Path("outputs/session_reports")


In [ ]:
# Pure timing and measurement helpers live in monitoring_logic.py.
# Position measurements are normalized proxies, not head rotation angles.
# Face directions refer to the mirrored preview.
# During calibration, face absence, multiple faces, and objects remain monitored.
# Press C to restart calibration while looking forward; press Q to finish.


In [ ]:
MODEL_DIR.mkdir(exist_ok=True)
SCREENSHOT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
session_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
session_start = time.monotonic()
calibration_samples = []
baseline = None
last_timestamp_ms = -1
trackers = {
    reason: EventTracker(
        FACE_ABSENCE_SECONDS if reason == "FACE ABSENT" else MIN_EVENT_DURATION,
        RECOVERY_PERIOD,
    )
    for reason in ("CELL PHONE", "BOOK", "CLOCK", "HEAD TURN", "HEAD PITCH",
                   "EYE GAZE", "FACE ABSENT", "MULTIPLE FACES")
}
counters = dict.fromkeys(trackers, 0)
events = []
current_event = None
cap = None
landmarker = None

try:
    model = YOLO(str(MODEL_DIR / "yolov8n.pt"))
    options = mp_vision.FaceLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=str(MODEL_DIR / "face_landmarker.task")),
        running_mode=mp_vision.RunningMode.VIDEO,
        num_faces=MAX_FACES,
        min_face_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    landmarker = mp_vision.FaceLandmarker.create_from_options(options)
    cap = cv2.VideoCapture(CAMERA_INDEX)
    if not cap.isOpened():
        raise RuntimeError("Unable to open the camera. Check camera access and CAMERA_INDEX.")

    while True:
        ret, frame = cap.read()
        if not ret:
            print("Unable to read from the camera. Ending session.")
            break
        now = time.monotonic()
        elapsed = now - session_start
        frame = cv2.flip(frame, 1)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        timestamp_ms = max(last_timestamp_ms + 1, int(elapsed * 1000))
        last_timestamp_ms = timestamp_ms
        result = landmarker.detect_for_video(mp_image, timestamp_ms)
        face_count = len(result.face_landmarks)
        head_status = pitch_status = gaze_status = "NO FACE"
        signals = dict.fromkeys(trackers, False)
        signals["FACE ABSENT"] = face_count == 0
        signals["MULTIPLE FACES"] = face_count > 1

        if face_count == 1:
            values = face_measurements(result.face_landmarks[0])
            head_status = pitch_status = gaze_status = "UNAVAILABLE"
            if values is not None:
                if baseline is None:
                    calibration_samples.append(values)
                    if len(calibration_samples) >= CALIBRATION_FRAMES:
                        baseline = calibration_baseline(calibration_samples)
                if baseline is not None:
                    head_status, pitch_status, gaze_status = classify_measurements(
                        values, baseline, HEAD_THRESHOLD, PITCH_THRESHOLD, GAZE_THRESHOLD,
                    )
                    signals["HEAD TURN"] = head_status != "FORWARD"
                    signals["HEAD PITCH"] = pitch_status != "LEVEL"
                    signals["EYE GAZE"] = gaze_status != "CENTER"
                else:
                    head_status = pitch_status = gaze_status = "CALIBRATING"
        elif face_count > 1:
            head_status = pitch_status = gaze_status = "MULTIPLE FACES"

        detected_objects = set()
        results = model(frame, classes=list(OBJECT_CLASSES),
                        conf=OBJECT_CONFIDENCE, verbose=False)
        for detection_result in results:
            for box in detection_result.boxes:
                name = OBJECT_CLASSES.get(int(box.cls[0]))
                if name is None:
                    continue
                detected_objects.add(name)
                x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                confidence = float(box.conf[0])
                allowed = name in ALLOWED_OBJECTS
                box_color = (0, 180, 0) if allowed else (0, 165, 255)
                cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)
                label = f"{name} {confidence:.2f}" + (" (ALLOWED)" if allowed else "")
                cv2.putText(frame, label, (x1, max(y1 - 10, 20)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, box_color, 2)
        for name in OBJECT_CLASSES.values():
            signals[name] = name in detected_objects and name not in ALLOWED_OBJECTS

        for reason, tracker in trackers.items():
            if tracker.update(signals[reason], now):
                counters[reason] += 1
        active_reasons = {reason for reason, tracker in trackers.items() if tracker.confirmed}

        if active_reasons:
            if current_event is None:
                image_path = SCREENSHOT_DIR / f"event_{session_id}_{len(events) + 1:04d}.jpg"
                saved = cv2.imwrite(str(image_path), frame)
                if not saved:
                    print(f"Warning: unable to save screenshot: {image_path}")
                current_event = {
                    "start_seconds": min(trackers[r].started_at for r in active_reasons) - session_start,
                    "confirmed_seconds": elapsed,
                    "reasons": sorted(active_reasons),
                    "screenshot": str(image_path) if saved else None,
                }
                events.append(current_event)
            current_event["reasons"] = sorted(set(current_event["reasons"]) | active_reasons)
            current_event["last_signal_seconds"] = max(
                trackers[r].last_signal_at for r in active_reasons
            ) - session_start
        elif current_event is not None:
            current_event["end_seconds"] = current_event["last_signal_seconds"]
            current_event["closed_seconds"] = elapsed
            current_event = None

        high_alert = any(
            tracker.confirmed and signals[reason] and tracker.duration(now) > HIGH_ALERT_SECONDS
            for reason, tracker in trackers.items()
        )
        pending = any(signals.values())
        status = ("HIGH ALERT" if high_alert else "FLAGGED FOR REVIEW" if active_reasons
                  else "OBSERVING" if pending else "NORMAL")
        color = (0, 0, 255) if active_reasons else (0, 165, 255) if pending else (0, 200, 0)
        calibration_text = ("Calibration complete" if baseline is not None else
                            f"Look forward: calibration {len(calibration_samples)}/{CALIBRATION_FRAMES}")
        lines = [
            # f"Head: {head_status} | Pitch: {pitch_status}",
            # f"Gaze: {gaze_status} | Faces: {face_count}",
            # "Objects: " + (", ".join(sorted(detected_objects)) or "NONE"),
            # f"Confirmed episodes: {len(events)} | {status}",
            # "Reasons: " + (", ".join(sorted(active_reasons)) or "NONE"),
            # calibration_text,
            # "Q: end session | C: recalibrate",
        ]
        for index, text in enumerate(lines):
            cv2.putText(frame, text, (15, 25 + index * 28),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
        cv2.imshow("Exam Monitoring - Review Assistant", frame)
        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break
        if key == ord("c"):
            calibration_samples.clear()
            baseline = None
            for reason in ("HEAD TURN", "HEAD PITCH", "EYE GAZE"):
                trackers[reason].reset()

finally:
    session_duration = time.monotonic() - session_start
    if cap is not None:
        cap.release()
    cv2.destroyAllWindows()
    if landmarker is not None:
        landmarker.close()
    if current_event is not None:
        current_event["end_seconds"] = current_event["last_signal_seconds"]
        current_event["closed_seconds"] = session_duration
        current_event["closed_by_session_end"] = True
    report_data = {
        "session_id": session_id,
        "duration_seconds": session_duration,
        "calibration_complete": baseline is not None,
        "baseline": baseline,
        "allowed_objects": sorted(ALLOWED_OBJECTS),
        "thresholds": {
            "minimum_event_seconds": MIN_EVENT_DURATION,
            "recovery_seconds": RECOVERY_PERIOD,
            "face_absence_seconds": FACE_ABSENCE_SECONDS,
            "high_alert_seconds": HIGH_ALERT_SECONDS,
            "head": HEAD_THRESHOLD, "pitch": PITCH_THRESHOLD,
            "gaze": GAZE_THRESHOLD, "object_confidence": OBJECT_CONFIDENCE,
        },
        "confirmed_episodes": len(events),
        "confirmed_reason_counts": counters,
        "events": events,
    }
    json_path = REPORT_DIR / f"session_report_{session_id}.json"
    json_path.write_text(json.dumps(report_data, indent=2), encoding="utf-8")
    report = "\n".join([
        "FINAL SESSION REPORT", "====================",
        f"Confirmed review episodes: {len(events)}",
        *[f"{reason}: {count}" for reason, count in counters.items()],
        f"Calibration complete: {baseline is not None}",
        f"Total session duration: {session_duration:.2f} seconds",
        "Counts represent confirmed episodes; overlapping reasons may share an episode.",
        "Flags require human review and do not establish cheating.",
    ])
    report_path = REPORT_DIR / f"session_report_{session_id}.txt"
    report_path.write_text(report, encoding="utf-8")
    print(report)
    print(f"Reports saved to: {report_path} and {json_path}")
